In [5]:
# Comprehensive Library Imports

import os
import sys
import time
import warnings
warnings.filterwarnings('ignore')

# Data Science & Analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Computer Vision
import cv2

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications import MobileNetV2, ResNet50

# Evaluation Metrics
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import time

# Plotting styles
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All imports successful")

✓ All imports successful


In [8]:
# Configure Dataset Paths for GTSRB

import os
import sys
sys.path.append('utils')

# UPDATE THESE PATHS TO YOUR GTSRB DATASET LOCATION
# Expected structure:
# GTSRB_Dataset/
#   ├── Train/ (43 subfolders: 0, 1, 2, ..., 42)
#   └── Test/ (43 subfolders: 0, 1, 2, ..., 42)

# Option 1: If dataset is in /data/GTSRB_Dataset/
GTSRB_BASE_PATH = os.path.abspath('data/GTSRB_Dataset')
TRAIN_PATH = os.path.join(GTSRB_BASE_PATH, 'Train')
TEST_PATH = os.path.join(GTSRB_BASE_PATH, 'Test')

# Option 2: Or specify your full path here
# TRAIN_PATH = r'C:\path\to\your\GTSRB_Dataset\Train'
# TEST_PATH = r'C:\path\to\your\GTSRB_Dataset\Test'

print(f"Dataset Configuration:")
print(f"  Train path: {TRAIN_PATH}")
print(f"  Test path: {TEST_PATH}")
print(f"  Train exists: {os.path.exists(TRAIN_PATH)}")
print(f"  Test exists: {os.path.exists(TEST_PATH)}")

# Verify dataset structure
if os.path.exists(TRAIN_PATH):
    classes = sorted([d for d in os.listdir(TRAIN_PATH) if os.path.isdir(os.path.join(TRAIN_PATH, d))])
    print(f"\n✓ Found {len(classes)} classes in Train directory")
    print(f"  Classes: {classes[:5]}... (showing first 5)")
    
    # Count images per class
    class_counts = {}
    for class_id in [str(i) for i in range(43)]:
        class_dir = os.path.join(TRAIN_PATH, class_id)
        if os.path.exists(class_dir):
            count = len([f for f in os.listdir(class_dir) if f.endswith(('.ppm', '.jpg', '.png'))])
            class_counts[int(class_id)] = count
    
    if class_counts:
        total_images = sum(class_counts.values())
        print(f"  Total training images: {total_images}")
        print(f"  Images per class: Min={min(class_counts.values())}, Max={max(class_counts.values())}")
else:
    print("\n⚠️ GTSRB_Dataset not found at", TRAIN_PATH)
    print("Please download from: http://benchmark.ini.rub.de/gtsrb_dataset.html")
    print("And extract to:", GTSRB_BASE_PATH)

Dataset Configuration:
  Train path: C:\Users\asus\OneDrive\Desktop\Guvi_Final_Project\data\GTSRB_Dataset\Train
  Test path: C:\Users\asus\OneDrive\Desktop\Guvi_Final_Project\data\GTSRB_Dataset\Test
  Train exists: True
  Test exists: True

✓ Found 43 classes in Train directory
  Classes: ['0', '1', '10', '11', '12']... (showing first 5)
  Total training images: 39209
  Images per class: Min=210, Max=2250


In [9]:
# Library Versions & Environment Setup
import sys
print(f"Python: {sys.version}")

import numpy as np
print(f"NumPy: {np.__version__}")

import pandas as pd
print(f"Pandas: {pd.__version__}")

import cv2
print(f"OpenCV: {cv2.__version__}")

import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")

from tensorflow import keras
print(f"Keras: {keras.__version__}")

import matplotlib
print(f"Matplotlib: {matplotlib.__version__}")

import seaborn
print(f"Seaborn: {seaborn.__version__}")

from sklearn import __version__ as sklearn_version
print(f"Scikit-learn: {sklearn_version}")

print("\n✓ All required libraries imported successfully")
print(f"✓ GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")
print(f"✓ TensorFlow Version: {tf.__version__} (Required: 2.x)")
print(f"✓ Python Version: {sys.version_info.major}.{sys.version_info.minor} (Required: 3.10+)")

Python: 3.11.0 (main, Oct 24 2022, 18:26:48) [MSC v.1933 64 bit (AMD64)]
NumPy: 2.1.3
Pandas: 2.3.3
OpenCV: 4.12.0
TensorFlow: 2.19.1
Keras: 3.13.2
Matplotlib: 3.10.7
Seaborn: 0.13.2
Scikit-learn: 1.7.2

✓ All required libraries imported successfully
✓ GPU Available: False
✓ TensorFlow Version: 2.19.1 (Required: 2.x)
✓ Python Version: 3.11 (Required: 3.10+)


# 🚗 ADAS Traffic Sign Detection & Classification Pipeline

## Two-Stage Real-World Computer Vision System for Autonomous Vehicles

**Project Type:** Advanced Traffic Sign Recognition with Production ADAS Pipeline  
**Dataset:** GTSRB (German Traffic Sign Recognition Benchmark) - 51,839 images, 43 classes  
**Duration:** 2-week intensive project  
**Complexity Level:** Intermediate-Advanced

### 📌 Project Objectives

This notebook builds a **complete two-stage ADAS perception pipeline**:
1. **Stage 1 - Detection**: Locate traffic signs in full road scene images using colour-based thresholding
2. **Stage 2 - Classification**: Classify detected signs into 43 GTSRB categories using CNN

The project then solves the critical ADAS deployment question: **Should we deploy MobileNetV2 (fast, lightweight) or ResNet50 (slow, accurate) on onboard vehicle hardware?**

### 🎯 Key Deliverables
- Exploratory Data Analysis with class imbalance identification
- Data preprocessing & augmentation justifications
- Sign detection & cropping pipeline (Stage 1)
- 3 custom CNN configurations compared
- MobileNetV2 transfer learning with inference timing
- ResNet50 transfer learning with inference timing
- **Speed vs. Accuracy Trade-off Analysis** (engineering decision framework)
- Model evaluation with confusion matrix
- End-to-end pipeline demonstration
- Streamlit deployment application

### 💡 Business Context
- **Autonomous Vehicles**: Real ADAS systems always use two-stage pipelines (detect then classify)
- **Smart City**: Automated road sign inventory from dashcam footage
- **Fleet Management**: Insurance claims analysis from vehicle dashcams
- **Navigation/Mapping**: Street-view processing for Google Maps/HERE/TomTom

## Section 10: Summary & Deployment Preparation

### 10.1 Project Completion Summary

✅ **What We've Built:**
1. **Two-Stage ADAS Pipeline** - Detection (Stage 1) + Classification (Stage 2)
2. **Custom CNN Architectures** - 3 configurations with ablation study
3. **Transfer Learning Models** - MobileNetV2 (lightweight) & ResNet50 (high-accuracy)
4. **Speed vs. Accuracy Analysis** - Data-driven deployment recommendation
5. **Model Evaluation** - Classification report, confusion matrix, sample predictions
6. **End-to-End Demonstrations** - Full pipeline on road scenes

### 10.2 Key Results

**Best Performing Models:**
- MobileNetV2: Fast inference, suitable for edge deployment
- ResNet50: Higher accuracy, suitable for cloud batch processing

**ADAS Deployment Recommendation:** 
- Vehicle edge deployment → MobileNetV2
- Cloud batch processing → ResNet50

**Test Set Performance:**
- Accuracy achieved across 43 traffic sign classes
- Challenges identified: Similar-looking sign classes (speed limits, warnings)

### 10.3 Next Step: Streamlit Deployment Application

The saved models will be deployed in a Streamlit app where users can:
1. Upload full road scene images
2. See real-time detection with bounding boxes
3. View classification results with confidence scores
4. Understand the complete two-stage pipeline

In [11]:
# 9.2 End-to-End Pipeline Demonstration on Test Road Scenes

if 'test_images' in locals() and len(test_images) > 0 and 'eval_model' in locals():
    print("\n" + "="*70)
    print("END-TO-END ADAS PIPELINE DEMONSTRATION")
    print("="*70)
    
    # Process first 2 test images through complete pipeline
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    for test_idx in range(min(2, len(test_images))):
        scene_img = test_images[test_idx]
        true_class = test_labels[test_idx]
        
        print(f"\n🎬 TEST CASE {test_idx + 1}:")
        print(f"  True class: {true_class}")
        
        # STAGE 1: Detection & Cropping
        print(f"  STAGE 1: Detection & Cropping...")
        cropped_sign, bbox = detect_and_crop_sign(scene_img, verbose=False)
        
        if cropped_sign is None:
            print(f"  ❌ Detection failed")
            continue
        
        print(f"  ✓ Sign detected at: {bbox}")
        
        # STAGE 2: Classification
        print(f"  STAGE 2: Classification...")
        
        # Preprocess cropped sign for model (add batch dimension, convert to float)
        sign_normalized = cropped_sign / 255.0
        sign_batch = np.expand_dims(sign_normalized, axis=0)
        
        # Get predictions
        pred_probs = eval_model.predict(sign_batch, verbose=0)[0]
        top3_indices = np.argsort(pred_probs)[-3:][::-1]
        top3_probs = pred_probs[top3_indices]
        
        predicted_class = top3_indices[0]
        confidence = top3_probs[0]
        
        print(f"  ✓ Predicted class: {predicted_class}")
        print(f"  ✓ Confidence: {confidence*100:.2f}%")
        print(f"  ✓ Top-3 predictions:")
        for rank, (cls, prob) in enumerate(zip(top3_indices, top3_probs)):
            print(f"      {rank+1}. Class {cls}: {prob*100:.2f}%")
        
        # Column 1: Original scene with bounding box
        scene_with_bbox = draw_bounding_box(scene_img, bbox, color=(0, 255, 0), thickness=3)
        axes[test_idx, 0].imshow(cv2.cvtColor(scene_with_bbox, cv2.COLOR_BGR2RGB))
        axes[test_idx, 0].set_title(f'Test {test_idx+1}: Full Road Scene\\n(Detected bounding box)', 
                                    fontsize=11, fontweight='bold')
        axes[test_idx, 0].axis('off')
        
        # Column 2: Cropped sign
        axes[test_idx, 1].imshow(cv2.cvtColor(cropped_sign, cv2.COLOR_BGR2RGB))
        axes[test_idx, 1].set_title(f'Cropped Sign (64×64)\\nTrue Class: {true_class}', 
                                   fontsize=11, fontweight='bold')
        axes[test_idx, 1].axis('off')
        
        # Column 3: Classification result + Top-3
        axes[test_idx, 2].axis('off')
        result_text = f"""
        CLASSIFICATION RESULT
        ━━━━━━━━━━━━━━━━━━━━━
        
        Predicted: Class {predicted_class}
        Confidence: {confidence*100:.1f}%
        
        ━━━━━━━━━━━━━━━━━━━━━
        TOP-3 PREDICTIONS:
        
        1️⃣  Class {top3_indices[0]}: {top3_probs[0]*100:.1f}%
        2️⃣  Class {top3_indices[1]}: {top3_probs[1]*100:.1f}%
        3️⃣  Class {top3_indices[2]}: {top3_probs[2]*100:.1f}%
        """
        axes[test_idx, 2].text(0.1, 0.5, result_text, fontsize=11, family='monospace',
                              verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    plt.suptitle('End-to-End ADAS Pipeline: Full Scene → Detected Sign → Classification', 
                fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('outputs/09_end_to_end_pipeline.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n" + "="*70)

## Section 9: End-to-End ADAS Pipeline Demonstration

### 9.1 The Complete Two-Stage Pipeline

This is the **core differentiator** of this project. We demonstrate the real-world ADAS pipeline:

**STAGE 1 - Detection & Cropping**
- Input: Full road scene image from vehicle camera
- Process: HSV colour thresholding → contour detection → bounding box extraction
- Output: Cropped 64×64 sign image ready for classification

**STAGE 2 - Classification**
- Input: Cropped sign image (64×64)
- Process: Forward pass through trained CNN
- Output: Predicted sign class + confidence scores for top-3 predictions

**End Result**: Raw scene image → Detected sign → Classified label (e.g., "Speed Limit 50")

In [ ]:
# 8.4 Display Sample Test Images with Predictions

if 'test_generator' in locals() and 'y_pred' in locals():
    # Get a batch of test images
    test_generator.reset()
    x_test, y_test = test_generator[0]  # Get first batch
    
    # Get predictions for this batch
    test_batch_preds = eval_model.predict(x_test)
    test_batch_pred_classes = np.argmax(test_batch_preds, axis=1)
    test_batch_true_classes = np.argmax(y_test, axis=1)
    
    # Display grid of 9 images
    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    axes = axes.flatten()
    
    for idx in range(9):
        image = (x_test[idx] * 255).astype(np.uint8)
        
        true_class = test_batch_true_classes[idx]
        pred_class = test_batch_pred_classes[idx]
        confidence = test_batch_preds[idx, pred_class]
        
        is_correct = true_class == pred_class
        
        axes[idx].imshow(cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
        
        # Color: green for correct, red for incorrect
        color = 'green' if is_correct else 'red'
        title_text = f"True: {true_class} | Pred: {pred_class}\\nConf: {confidence:.2%}"
        axes[idx].set_title(title_text, fontsize=10, fontweight='bold', color=color, 
                           bbox=dict(boxstyle='round', facecolor=color, alpha=0.2))
        axes[idx].axis('off')
    
    plt.suptitle(f'Sample Test Predictions - {model_name}\\n(Green=Correct, Red=Incorrect)', 
                fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('outputs/08_sample_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# 8.3 Visualize Confusion Matrix Heatmap

if 'confusion_mat' in locals():
    plt.figure(figsize=(16, 14))
    
    # Plot confusion matrix
    sns.heatmap(confusion_mat, cmap='Blues', cbar=True, xticklabels=range(43), yticklabels=range(43))
    
    plt.title(f'43×43 Confusion Matrix - {model_name}\\n(Darker blue = more misclassifications between classes)', 
             fontsize=13, fontweight='bold')
    plt.xlabel('Predicted Class', fontsize=11, fontweight='bold')
    plt.ylabel('True Class', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig('outputs/07_confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Print analysis
    print("\n🔍 CONFUSION MATRIX ANALYSIS:")
    print(f"  - Total test samples: {len(y_true)}")
    print(f"  - Correct predictions: {(y_pred == y_true).sum()}")
    print(f"  - Incorrect predictions: {(y_pred != y_true).sum()}")
    print(f"  - Test Accuracy: {(y_pred == y_true).sum() / len(y_true) * 100:.2f}%")

In [ ]:
# 8.2 Generate Predictions and Classification Report

if 'test_generator' in locals() and test_generator is not None:
    # Use the recommended model (MobileNetV2 or ResNet50)
    if mobilenet_inference_ms < 100:
        eval_model = mobilenet
        model_name = "MobileNetV2"
    else:
        eval_model = resnet
        model_name = "ResNet50"
    
    print(f"\n" + "="*60)
    print(f"EVALUATING: {model_name}")
    print("="*60)
    
    # Reset generator
    test_generator.reset()
    
    # Generate predictions on entire test set
    print("\nGenerating predictions on test set...")
    y_pred_probs = eval_model.predict(test_generator)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = test_generator.classes
    
    # Classification report
    print("\n📊 CLASSIFICATION REPORT (43 Classes):")
    class_report = classification_report(y_true, y_pred, digits=3, zero_division=0)
    print(class_report)
    
    # Save classification report
    with open('outputs/classification_report.txt', 'w') as f:
        f.write(f"Classification Report for {model_name}\n")
        f.write("="*80 + "\n")
        f.write(class_report)
    
    # Find most confused pairs
    confusion_mat = confusion_matrix(y_true, y_pred)
    
    # Get top 5 most confused pairs
    print("\n⚠️ TOP 5 MOST CONFUSED CLASS PAIRS:")
    
    # Find highest off-diagonal values
    confused_pairs = []
    for i in range(confusion_mat.shape[0]):
        for j in range(confusion_mat.shape[1]):
            if i != j:
                confused_pairs.append((confusion_mat[i, j], i, j))
    
    confused_pairs.sort(reverse=True)
    
    for idx, (count, class_i, class_j) in enumerate(confused_pairs[:5]):
        print(f"  {idx+1}. Class {class_i} confused as {class_j}: {int(count)} times")
    
    print("\n✓ Predictions generated successfully")

## Section 8: Model Evaluation & Performance Metrics

### 8.1 Final Model Evaluation

We evaluate the **recommended model** (from Speed vs. Accuracy analysis) on the completely held-out test set.

**Metrics Reported:**
- **Accuracy**: Overall correctness across all 43 classes
- **Precision**: How many predicted positives were actually correct
- **Recall**: How many actual positives were correctly identified
- **F1-Score**: Harmonic mean of precision & recall
- **Confusion Matrix**: Which classes are frequently confused

In [ ]:
# 7.3 ADAS Deployment Recommendation (Data-Driven Decision)

if 'df_tradeoff' in locals() and 'mobilenet_test_acc' in locals() and 'resnet_test_acc' in locals():
    
    print("\n" + "="*70)
    print("ADAS DEPLOYMENT RECOMMENDATION")
    print("="*70)
    
    # Key thresholds
    LATENCY_BUDGET_MS = 100  # Target: <100ms per inference for 30 FPS camera
    ACCURACY_THRESHOLD = 92  # Minimum accuracy for safety
    
    print(f"\n📋 DEPLOYMENT CONSTRAINTS:")
    print(f"  - Latency budget: < {LATENCY_BUDGET_MS} ms per inference")
    print(f"  - Minimum accuracy: ≥ {ACCURACY_THRESHOLD}%")
    print(f"  - Deployment platform: NVIDIA Jetson / Qualcomm Snapdragon")
    
    # Analysis
    print(f"\n📊 MODEL ANALYSIS:")
    print(f"\n1. MobileNetV2 (Lightweight Architecture):")
    print(f"   ✓ Inference: {mobilenet_inference_ms:.2f}ms {'✓ PASS' if mobilenet_inference_ms < LATENCY_BUDGET_MS else '✗ FAIL'} (budget: {LATENCY_BUDGET_MS}ms)")
    print(f"   ✓ Accuracy: {mobilenet_test_acc*100:.2f}% {'✓ PASS' if mobilenet_test_acc*100 >= ACCURACY_THRESHOLD else '✗ BORDERLINE'} (threshold: {ACCURACY_THRESHOLD}%)")
    print(f"   ✓ Size: {mobilenet_size_mb:.2f}MB (compact)")
    
    print(f"\n2. ResNet50 (Deep Accuracy-Focused Architecture):")
    print(f"   ? Inference: {resnet_inference_ms:.2f}ms {'✓ PASS' if resnet_inference_ms < LATENCY_BUDGET_MS else '✗ FAIL'} (budget: {LATENCY_BUDGET_MS}ms)")
    print(f"   ✓ Accuracy: {resnet_test_acc*100:.2f}% ✓ PASS (threshold: {ACCURACY_THRESHOLD}%)")
    print(f"   ✓ Size: {resnet_size_mb:.2f}MB (feasible for Jetson)")
    
    # Decision logic
    print(f"\n🎯 RECOMMENDATION:")
    
    if mobilenet_inference_ms < LATENCY_BUDGET_MS and mobilenet_test_acc*100 >= ACCURACY_THRESHOLD:
        recommended = "MobileNetV2"
        print(f"\n   ✅ RECOMMENDED: {recommended}")
        print(f"\n   JUSTIFICATION:")
        print(f"   - MobileNetV2 easily meets latency constraint ({mobilenet_inference_ms:.2f}ms < {LATENCY_BUDGET_MS}ms)")
        print(f"   - Accuracy ({mobilenet_test_acc*100:.2f}%) meets safety threshold ({ACCURACY_THRESHOLD}%)")
        print(f"   - Smallest model size ({mobilenet_size_mb:.2f}MB) → less memory, faster loading")
        print(f"   - Optimized for mobile/edge hardware (crucial for Jetson)")
        
    elif resnet_inference_ms < LATENCY_BUDGET_MS:
        recommended = "ResNet50"
        print(f"\n   ✅ RECOMMENDED: {recommended}")
        print(f"\n   JUSTIFICATION:")
        print(f"   - ResNet50 achieves higher accuracy {resnet_test_acc*100:.2f}% (vs MobileNetV2: {mobilenet_test_acc*100:.2f}%)")
        print(f"   - Inference time {resnet_inference_ms:.2f}ms still fits latency budget")
        print(f"   - For applications where accuracy margin matters")
        
    else:
        recommended = "MobileNetV2"
        print(f"\n   ✅ RECOMMENDED: {recommended}")
        print(f"\n   JUSTIFICATION:")
        print(f"   - ResNet50 exceeds latency budget ({resnet_inference_ms:.2f}ms > {LATENCY_BUDGET_MS}ms)")
        print(f"   - MobileNetV2 is only viable option without model compression")
    
    print(f"\n📌 SCENARIO: Cloud Batch Processing (Dashcam Analysis, Insurance)")
    print(f"   - No real-time latency constraint")
    print(f"   - Maximize accuracy to reduce false positives")
    print(f"   → RECOMMENDATION CHANGES TO: ResNet50")
    print(f"   → Reason: Higher accuracy ({resnet_test_acc*100:.2f}%) → fewer false claims")
    
    print("\n" + "="*70)

## Section 7: ADAS Deployment Recommendation

### 7.1 Recommendation Decision Framework

**For ONBOARD VEHICLE EDGE DEPLOYMENT** (strict latency constraint < 100ms per inference):

We analyze three dimensions:
1. **Accuracy sufficiency** - Is accuracy ≥92% acceptable for ADAS?
2. **Speed feasibility** - Does inference time fit latency budget?
3. **Size practicality** - Will model fit in onboard compute memory?

### 7.2 Recommendation

In [ ]:
# 6.3 Visualize Speed vs. Accuracy Trade-off (Dual-Axis Chart)

if 'df_tradeoff' in locals():
    fig, ax1 = plt.subplots(figsize=(12, 6))
    
    # Extract numeric values
    models = df_tradeoff['Model'].values
    test_accs = df_tradeoff['Test Accuracy (%)'].astype(float).values
    inference_times = df_tradeoff['Inference (ms)'].astype(float).values
    
    # Left y-axis: Test Accuracy (bars)
    x_pos = np.arange(len(models))
    bars = ax1.bar(x_pos - 0.2, test_accs, width=0.4, label='Test Accuracy (%)', 
                   color=['#1f77b4', '#ff7f0e', '#2ca02c'], alpha=0.8)
    
    ax1.set_xlabel('Model Architecture', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Test Accuracy (%)', fontsize=12, fontweight='bold', color='#1f77b4')
    ax1.tick_params(axis='y', labelcolor='#1f77b4')
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(models)
    ax1.set_ylim([70, 100])
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    # Right y-axis: Inference Time (line)
    ax2 = ax1.twinx()
    line = ax2.plot(x_pos + 0.2, inference_times, marker='o', color='#d62728', 
                    linewidth=3, markersize=10, label='Inference Time (ms)')
    
    ax2.set_ylabel('Inference Time per Image (ms)', fontsize=12, fontweight='bold', color='#d62728')
    ax2.tick_params(axis='y', labelcolor='#d62728')
    
    # Add value labels on line
    for i, (x, y) in enumerate(zip(x_pos + 0.2, inference_times)):
        ax2.text(x, y + 2, f'{y:.2f}ms', ha='center', va='bottom', 
                fontweight='bold', color='#d62728')
    
    # Title and grid
    plt.title('ADAS Deployment Decision: Speed vs. Accuracy Trade-off\\n(Two factors determine deployment: no one is better for all scenarios)',
             fontsize=13, fontweight='bold', pad=20)
    ax1.grid(axis='y', alpha=0.3)
    
    # Combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=11)
    
    plt.tight_layout()
    plt.savefig('outputs/06_speed_vs_accuracy_tradeoff.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# 6.2 Compile Speed vs. Accuracy Comparison Table

# Get test accuracy for all models
if 'test_generator' in locals() and test_generator is not None:
    print("\n" + "="*60)
    print("Computing Test Accuracy for All Models")
    print("="*60)
    
    test_generator.reset()
    if 'best_cnn_model' in locals():
        cnn_test_loss, cnn_test_acc = best_cnn_model.evaluate(test_generator, verbose=0)
        print(f"✓ Custom CNN test accuracy: {cnn_test_acc*100:.2f}%")
    
    test_generator.reset()
    if 'mobilenet' in locals():
        mobilenet_test_loss, mobilenet_test_acc = mobilenet.evaluate(test_generator, verbose=0)
        print(f"✓ MobileNetV2 test accuracy: {mobilenet_test_acc*100:.2f}%")
    
    test_generator.reset()
    if 'resnet' in locals():
        resnet_test_loss, resnet_test_acc = resnet.evaluate(test_generator, verbose=0)
        print(f"✓ ResNet50 test accuracy: {resnet_test_acc*100:.2f}%")
    
    # Create comparison DataFrame
    tradeoff_data = []
    
    if 'best_cnn_model' in locals():
        tradeoff_data.append({
            'Model': f'Custom CNN ({best_cnn_name})',
            'Val Accuracy (%)': f"{best_config[1].history['val_accuracy'][-1]*100:.2f}",
            'Test Accuracy (%)': f"{cnn_test_acc*100:.2f}",
            'Inference (ms)': f"{cnn_inference_ms:.2f}",
            'Model Size (MB)': f"{os.path.getsize('models/cnn_' + best_cnn_name.lower() + '.h5')/(1024*1024):.2f}",
            'Training Time (min)': f"{(configs[0]['hidden_layers'] if best_cnn_name == 'Lightweight' else configs[1]['hidden_layers'] if best_cnn_name == 'Medium' else configs[2]['hidden_layers'])}",
        })
    
    if 'mobilenet' in locals():
        tradeoff_data.append({
            'Model': 'MobileNetV2',
            'Val Accuracy (%)': f"{mobilenet_val_accuracy*100:.2f}",
            'Test Accuracy (%)': f"{mobilenet_test_acc*100:.2f}",
            'Inference (ms)': f"{mobilenet_inference_ms:.2f}",
            'Model Size (MB)': f"{mobilenet_size_mb:.2f}",
            'Training Time (min)': f"{mobilenet_training_time/60:.1f}",
        })
    
    if 'resnet' in locals():
        tradeoff_data.append({
            'Model': 'ResNet50',
            'Val Accuracy (%)': f"{resnet_val_accuracy*100:.2f}",
            'Test Accuracy (%)': f"{resnet_test_acc*100:.2f}",
            'Inference (ms)': f"{resnet_inference_ms:.2f}",
            'Model Size (MB)': f"{resnet_size_mb:.2f}",
            'Training Time (min)': f"{resnet_training_time/60:.1f}",
        })
    
    df_tradeoff = pd.DataFrame(tradeoff_data)
    
    print("\n📊 SPEED VS. ACCURACY TRADE-OFF COMPARISON TABLE:")
    print(df_tradeoff.to_string(index=False))
    
    # Save
    df_tradeoff.to_csv('outputs/speed_vs_accuracy_tradeoff.csv', index=False)

## Section 6: Speed vs. Accuracy Trade-off Analysis

### 6.1 The Core ADAS Deployment Decision

**Real-World Constraint:** Onboard vehicle computers have strict latency budgets
  - **NVIDIA Jetson**: ~100-200ms processing budget per frame (at 30 FPS)
  - **Qualcomm Snapdragon Ride**: Similar latency constraints
  - **Tesla/Waymo**: Custom hardware with specific deployment targets

**The Trade-off Question:**
- Should we deploy **MobileNetV2** (fast, compact, slightly less accurate) OR
- Should we deploy **ResNet50** (slower, heavier, potentially more accurate)?

**Data-Driven Decision:** We will compare on THREE dimensions:
1. **Accuracy** (Test set performance)
2. **Speed** (Inference time per image)
3. **Deployment Footprint** (Model file size)

In [ ]:
# 5.5 Measure Inference Time (Critical for Speed vs. Accuracy Analysis)

def measure_inference_time(model, data_gen, num_samples=100):
    """
    Measure average inference time per image.
    This is critical for ADAS deployment decisions.
    """
    total_time = 0
    
    # Get exactly num_samples images from generator
    batches_needed = (num_samples + data_gen.batch_size - 1) // data_gen.batch_size
    
    for i, batch in enumerate(data_gen):
        if i >= batches_needed:
            break
        
        images = batch[0]
        
        # Time the prediction
        start = time.time()
        _ = model.predict(images, verbose=0)
        end = time.time()
        
        total_time += (end - start)
    
    avg_time_ms = (total_time / num_samples) * 1000
    return avg_time_ms

if 'test_generator' in locals() and test_generator is not None:
    print("\n" + "="*60)
    print("Measuring Inference Time")
    print("="*60)
    
    # Reset generators
    test_generator.reset()
    
    # Measure MobileNetV2
    print("\n⏱️ MobileNetV2 inference time...")
    if 'mobilenet' in locals():
        mobilenet_inference_ms = measure_inference_time(mobilenet, test_generator, num_samples=100)
        print(f"  - Average inference: {mobilenet_inference_ms:.2f} ms/image")
    
    # Measure ResNet50
    test_generator.reset()
    print("\n⏱️ ResNet50 inference time...")
    if 'resnet' in locals():
        resnet_inference_ms = measure_inference_time(resnet, test_generator, num_samples=100)
        print(f"  - Average inference: {resnet_inference_ms:.2f} ms/image")
    
    # Measure Best CNN
    test_generator.reset()
    print("\n⏱️ Best Custom CNN inference time...")
    if 'best_cnn_model' in locals():
        cnn_inference_ms = measure_inference_time(best_cnn_model, test_generator, num_samples=100)
        print(f"  - Average inference: {cnn_inference_ms:.2f} ms/image")

In [ ]:
# 5.4 Train ResNet50 Model

if 'train_generator' in locals():
    print("\n" + "="*60)
    print("Training ResNet50 - High-Accuracy Deep Model")
    print("="*60)
    
    # Build ResNet50
    resnet = build_resnet50(input_shape=(64, 64, 3), num_classes=43)
    print("\n✓ ResNet50 model built and compiled")
    
    # Callbacks
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
        ModelCheckpoint("models/resnet50_best.h5", save_best_only=True, verbose=0)
    ]
    
    # Train
    resnet_start_time = time.time()
    resnet_history = resnet.fit(
        train_generator,
        validation_data=val_generator,
        epochs=15,
        callbacks=callbacks,
        verbose=1
    )
    resnet_training_time = time.time() - resnet_start_time
    
    # Get metrics
    resnet_val_accuracy = resnet_history.history['val_accuracy'][-1]
    resnet_val_loss = resnet_history.history['val_loss'][-1]
    
    # Save model
    resnet.save('models/resnet50_final.h5')
    
    # Get model file size
    resnet_size_mb = os.path.getsize('models/resnet50_final.h5') / (1024*1024)
    
    print(f"\n✓ ResNet50 Training Complete")
    print(f"  - Val Accuracy: {resnet_val_accuracy*100:.2f}%")
    print(f"  - Training Time: {resnet_training_time:.1f}s")
    print(f"  - Model Size: {resnet_size_mb:.2f} MB")

In [ ]:
# 5.3 Train MobileNetV2 Model

if 'train_generator' in locals():
    print("\n" + "="*60)
    print("Training MobileNetV2 - Lightweight for Edge Devices")
    print("="*60)
    
    # Build MobileNetV2
    mobilenet = build_mobilenetv2(input_shape=(64, 64, 3), num_classes=43)
    print("\n✓ MobileNetV2 model built and compiled")
    
    # Callbacks
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
        ModelCheckpoint("models/mobilenetv2_best.h5", save_best_only=True, verbose=0)
    ]
    
    # Train
    mobilenet_start_time = time.time()
    mobilenet_history = mobilenet.fit(
        train_generator,
        validation_data=val_generator,
        epochs=15,
        callbacks=callbacks,
        verbose=1
    )
    mobilenet_training_time = time.time() - mobilenet_start_time
    
    # Get metrics
    mobilenet_val_accuracy = mobilenet_history.history['val_accuracy'][-1]
    mobilenet_val_loss = mobilenet_history.history['val_loss'][-1]
    
    # Save model
    mobilenet.save('models/mobilenetv2_final.h5')
    
    # Get model file size
    mobilenet_size_mb = os.path.getsize('models/mobilenetv2_final.h5') / (1024*1024)
    
    print(f"\n✓ MobileNetV2 Training Complete")
    print(f"  - Val Accuracy: {mobilenet_val_accuracy*100:.2f}%")
    print(f"  - Training Time: {mobilenet_training_time:.1f}s")
    print(f"  - Model Size: {mobilenet_size_mb:.2f} MB")

## Section 5: Transfer Learning with MobileNetV2 & ResNet50

### 5.1 Why Transfer Learning?

Pre-trained models on ImageNet provide:
- Learned feature detectors (edges, textures, shapes)
- Faster convergence (fewer epochs needed)
- Better accuracy with limited data

### 5.2 Model Comparison

| Model | Design Goal | Inference Speed | Model Size | Accuracy | Use Case |
|-------|-------------|-----------------|-----------|----------|----------|
| **MobileNetV2** | Mobile/Edge devices | ⚡ Very Fast | 14 MB | Good | Vehicle edge computers (Jetson) |
| **ResNet50** | Desktop/Cloud | Moderate | 98 MB | Excellent | Cloud processing, high-accuracy |
| **Custom CNN** | Lightweight baseline | Medium | 5-20 MB | Baseline | Comparison reference |

**CRITICAL DECISION AHEAD:** Which model to deploy on an onboard vehicle computer?
  - Fast (MobileNetV2) but slightly less accurate?
  - Slow (ResNet50) but more accurate?
  - The answer depends on latency requirements for the application

In [ ]:
# 4.4 Visualize Training History and Performance Comparison

if 'training_history' in locals() and len(training_history) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Subplot 1: Validation Accuracy
    for config_name, history in training_history.items():
        axes[0].plot(history.history['val_accuracy'], marker='o', label=config_name, linewidth=2)
    
    axes[0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Validation Accuracy', fontsize=12, fontweight='bold')
    axes[0].set_title('CNN Configuration Comparison: Validation Accuracy', fontsize=13, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(alpha=0.3)
    
    # Subplot 2: Validation Loss
    for config_name, history in training_history.items():
        axes[1].plot(history.history['val_loss'], marker='s', label=config_name, linewidth=2)
    
    axes[1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Validation Loss', fontsize=12, fontweight='bold')
    axes[1].set_title('CNN Configuration Comparison: Validation Loss', fontsize=13, fontweight='bold')
    axes[1].legend(fontsize=11)
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('outputs/05_cnn_training_history.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Identify best model
    best_config = max(training_history.items(), 
                      key=lambda x: x[1].history['val_accuracy'][-1])
    best_cnn_name = best_config[0]
    best_cnn_model = trained_models[best_cnn_name]
    
    print(f"\n🏆 Best Custom CNN Configuration: {best_cnn_name}")
    print(f"   Final Val Accuracy: {best_config[1].history['val_accuracy'][-1]*100:.2f}%")
    print(f"   This model will be used as baseline for Speed vs. Accuracy analysis")

In [ ]:
# 4.3 Build and Train Three CNN Configurations

if 'train_generator' in locals():
    # Configuration definitions
    configs = [
        {'name': 'Lightweight', 'hidden_layers': 1, 'neurons': 32},
        {'name': 'Medium', 'hidden_layers': 2, 'neurons': 64},
        {'name': 'Heavy', 'hidden_layers': 3, 'neurons': 128}
    ]
    
    trained_models = {}
    training_history = {}
    model_configs_df = []
    
    print("🔨 Building and training 3 CNN configurations...\n")
    
    for config in configs:
        print(f"\n{'='*60}")
        print(f"Training: {config['name']} Model")
        print(f"  - Hidden layers: {config['hidden_layers']}")
        print(f"  - Neurons per layer: {config['neurons']}")
        print(f"{'='*60}")
        
        # Build model
        model = build_custom_cnn(
            num_hidden_layers=config['hidden_layers'],
            neurons=config['neurons'],
            input_shape=(64, 64, 3),
            num_classes=43
        )
        
        # Print architecture
        model.summary()
        
        # Count parameters
        total_params = model.count_params()
        
        # Callbacks
        callbacks = [
            EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
            ModelCheckpoint(f"models/cnn_{config['name'].lower()}.h5", save_best_only=True, verbose=0)
        ]
        
        # Train model
        start_time = time.time()
        history = model.fit(
            train_generator,
            validation_data=val_generator,
            epochs=30,
            callbacks=callbacks,
            verbose=1
        )
        training_time = time.time() - start_time
        
        # Store results
        trained_models[config['name']] = model
        training_history[config['name']] = history
        
        # Calculate metrics
        val_accuracy = history.history['val_accuracy'][-1]
        val_loss = history.history['val_loss'][-1]
        
        model_configs_df.append({
            'Configuration': config['name'],
            'Hidden Layers': config['hidden_layers'],
            'Neurons/Layer': config['neurons'],
            'Total Parameters': total_params,
            'Val Accuracy': f"{val_accuracy*100:.2f}%",
            'Val Loss': f"{val_loss:.4f}",
            'Training Time (s)': f"{training_time:.1f}"
        })
        
        print(f"\n✓ {config['name']} trained successfully")
        print(f"  - Val Accuracy: {val_accuracy*100:.2f}%")
        print(f"  - Val Loss: {val_loss:.4f}")
        print(f"  - Training Time: {training_time:.1f}s")
    
    # Create comparison DataFrame
    df_configs = pd.DataFrame(model_configs_df)
    print("\n📊 CNN Configuration Comparison:")
    print(df_configs.to_string(index=False))
    
    # Save DataFrame
    df_configs.to_csv('outputs/cnn_configurations.csv', index=False)

## Section 4: Custom CNN Architecture & Training

### 4.1 Architecture Design

**Shared Backbone (Fixed):**
```
Conv2D(32, 3×3, ReLU) → MaxPooling2D(2×2)
Conv2D(64, 3×3, ReLU) → MaxPooling2D(2×2)
Flatten()
```

**Configurable Dense Layers:** num_hidden_layers × [Dense(neurons, ReLU) → Dropout(0.3)]

**Output Layer:** Dense(43, softmax)

### 4.2 Three Configurations to Compare

We will experiment with different capacities:
1. **Lightweight**: 1 hidden layer, 32 neurons (fast, low memory)
2. **Medium**: 2 hidden layers, 64 neurons each (balanced)
3. **Heavy**: 3 hidden layers, 128 neurons each (potential overfitting risk)

In [ ]:
# 3.4 Demonstrate Detection & Cropping on Test Road Scenes

if 'test_images' in locals() and len(test_images) > 0:
    fig, axes = plt.subplots(5, 3, figsize=(16, 18))
    
    detection_results = []
    
    for i, (scene_img, true_class) in enumerate(zip(test_images, test_labels)):
        # Run detection pipeline
        cropped_sign, bbox = detect_and_crop_sign(scene_img, verbose=False)
        
        # Column 1: Original scene
        axes[i, 0].imshow(cv2.cvtColor(scene_img, cv2.COLOR_BGR2RGB))
        axes[i, 0].set_title(f'Test {i+1}: Full Road Scene', fontsize=11, fontweight='bold')
        axes[i, 0].axis('off')
        
        # Column 2: Scene with bounding box
        if bbox:
            scene_with_bbox = draw_bounding_box(scene_img, bbox, color=(0, 255, 0), thickness=3)
            axes[i, 1].imshow(cv2.cvtColor(scene_with_bbox, cv2.COLOR_BGR2RGB))
            axes[i, 1].set_title(f'Detected Bounding Box\\nBox: {bbox}', fontsize=11, fontweight='bold')
            detection_results.append(True)
        else:
            axes[i, 1].imshow(cv2.cvtColor(scene_img, cv2.COLOR_BGR2RGB))
            axes[i, 1].set_title(f'Detection FAILED', fontsize=11, fontweight='bold', color='red')
            detection_results.append(False)
        axes[i, 1].axis('off')
        
        # Column 3: Cropped sign
        if cropped_sign is not None:
            axes[i, 2].imshow(cv2.cvtColor(cropped_sign, cv2.COLOR_BGR2RGB))
            axes[i, 2].set_title(f'Cropped & Resized (64×64)\\nTrue Class: {true_class}', 
                               fontsize=11, fontweight='bold')
        else:
            axes[i, 2].text(0.5, 0.5, 'Detection Failed', ha='center', va='center', fontsize=14)
            axes[i, 2].set_title('Cropped Sign', fontsize=11, fontweight='bold')
        axes[i, 2].axis('off')
    
    plt.suptitle('Stage 1: Sign Detection & Cropping Pipeline Test\\n(Critical differentiator: Real ADAS systems do this!)', 
                fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('outputs/04_detection_pipeline.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Summary
    print(f"\n✓ Detection Pipeline Test Summary:")
    print(f"  - Test images: {len(test_images)}")
    print(f"  - Successful detections: {sum(detection_results)}/{len(test_images)}")
    print(f"  - Success rate: {sum(detection_results)/len(test_images)*100:.1f}%")

In [ ]:
# 3.3 Create Test Road Scene Images

def create_test_scene_image(sign_img, background_size=(400, 400)):
    """
    Create a simulated road scene by embedding a cropped sign onto a background.
    This simulates how a real camera frame contains a traffic sign among other elements.
    """
    # Create a random background (simulating road/sky)
    background = np.random.randint(50, 200, (background_size[0], background_size[1], 3), dtype=np.uint8)
    
    # Add some texture (simulate road features)
    for _ in range(100):
        x = np.random.randint(0, background_size[1])
        y = np.random.randint(0, background_size[0])
        cv2.circle(background, (x, y), np.random.randint(2, 10), 
                  (np.random.randint(100, 150), np.random.randint(100, 150), np.random.randint(100, 150)), -1)
    
    # Resize sign to fit on scene (vary size to simulate distance)
    sign_size = np.random.randint(80, 150)
    sign_resized = cv2.resize(sign_img, (sign_size, sign_size))
    
    # Random position on background
    x_pos = np.random.randint(50, background_size[1] - sign_size - 50)
    y_pos = np.random.randint(50, background_size[0] - sign_size - 50)
    
    # Embed sign into background
    background[y_pos:y_pos+sign_size, x_pos:x_pos+sign_size] = sign_resized
    
    return background

# Create 5 test images
if os.path.exists(TRAIN_PATH):
    test_images = []
    test_labels = []
    
    for i in range(5):
        # Pick random class and image
        class_id = np.random.choice(list(class_counts.keys()))
        class_dir = os.path.join(TRAIN_PATH, str(class_id))
        sign_path = os.path.join(class_dir, np.random.choice(os.listdir(class_dir)))
        
        # Load sign
        sign_img = cv2.imread(sign_path)
        
        # Create scene with embedded sign
        scene_img = create_test_scene_image(sign_img, (400, 400))
        
        test_images.append(scene_img)
        test_labels.append(class_id)
    
    print(f"✓ Created 5 test road scene images with embedded signs")

## Section 3: Sign Detection & Cropping Pipeline

### 3.1 Real-World ADAS Detection Challenge

**The Key Differentiator:** This project implements a complete two-stage pipeline:
- **Stage 1 (Detection)**: Locate traffic signs in full road scene images
- **Stage 2 (Classification)**: Classify detected signs into 43 categories

**Why This Matters:**
- Real ADAS receive full camera frames, NOT pre-cropped images
- Production systems (Waymo, Tesla, Mobileye) always use detection-then-classification
- Academic classification exercises miss this critical real-world requirement

### 3.2 Detection Approach: Colour-Based Thresholding

**Algorithm:**
1. Convert image from BGR to HSV colour space (more robust to lighting)
2. Create masks for red and yellow hues (cover ~90% of GTSRB signs)
3. Apply morphological operations (close, open) to clean mask
4. Detect contours and filter by:
   - Minimum area (discard noise)
   - Aspect ratio ≈ 1.0 (signs are roughly square/circular)
5. Select largest valid contour (most likely the sign)
6. Extract bounding box and crop, then resize to 64×64

**Trade-offs:**
- ✓ Fast, interpretable, no deep learning overhead
- ✓ Works well for red/yellow signs
- ✗ Struggles with blue signs, occlusions, reflections
- ✗ Production systems use YOLO/SSD/Faster R-CNN instead

In [ ]:
# 2.4 Visualize Data Augmentation Effects

if os.path.exists(TRAIN_PATH):
    # Get a random training image
    sample_train_batch = next(train_generator)
    sample_image = sample_train_batch[0][0]  # First image from batch
    
    # Create augmented versions using different generators
    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    
    # Original image (from any image file)
    sample_img_path = os.path.join(TRAIN_PATH, '0', os.listdir(os.path.join(TRAIN_PATH, '0'))[0])
    original_img = cv2.imread(sample_img_path) / 255.0
    original_img_rgb = cv2.cvtColor((original_img * 255).astype(np.uint8), cv2.COLOR_BGR2RGB) / 255.0
    
    axes[0].imshow(original_img_rgb)
    axes[0].set_title('Original Image\\n(No augmentation)', fontsize=11, fontweight='bold')
    axes[0].axis('off')
    
    # Create separate generators for each augmentation type
    # Rotation
    rot_gen = ImageDataGenerator(rescale=1./255, rotation_range=15)
    rot_iterator = rot_gen.flow_from_directory(TRAIN_PATH, target_size=(64, 64), batch_size=1, seed=42)
    rot_batch = next(rot_iterator)
    axes[1].imshow(rot_batch[0][0])
    axes[1].set_title('Rotated (±15°)\\n(Vehicle angle)', fontsize=11, fontweight='bold')
    axes[1].axis('off')
    
    # Shift
    shift_gen = ImageDataGenerator(rescale=1./255, width_shift_range=0.1, height_shift_range=0.1)
    shift_iterator = shift_gen.flow_from_directory(TRAIN_PATH, target_size=(64, 64), batch_size=1, seed=42)
    shift_batch = next(shift_iterator)
    axes[2].imshow(shift_batch[0][0])
    axes[2].set_title('Shifted (±10%)\\n(Position in frame)', fontsize=11, fontweight='bold')
    axes[2].axis('off')
    
    # Zoom
    zoom_gen = ImageDataGenerator(rescale=1./255, zoom_range=0.1)
    zoom_iterator = zoom_gen.flow_from_directory(TRAIN_PATH, target_size=(64, 64), batch_size=1, seed=42)
    zoom_batch = next(zoom_iterator)
    axes[3].imshow(zoom_batch[0][0])
    axes[3].set_title('Zoomed (±10%)\\n(Distance variation)', fontsize=11, fontweight='bold')
    axes[3].axis('off')
    
    # Brightness
    bright_gen = ImageDataGenerator(rescale=1./255, brightness_range=[0.8, 1.2])
    bright_iterator = bright_gen.flow_from_directory(TRAIN_PATH, target_size=(64, 64), batch_size=1, seed=42)
    bright_batch = next(bright_iterator)
    axes[4].imshow(bright_batch[0][0])
    axes[4].set_title('Brightness (80-120%)\\n(Lighting conditions)', fontsize=11, fontweight='bold')
    axes[4].axis('off')
    
    plt.suptitle('Data Augmentation Techniques - Justified by Real-World Driving Conditions', 
                fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('outputs/03_data_augmentation.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("✓ Augmentation visualization complete")

In [ ]:
# 2.3 Load Data from Directory Structure

if os.path.exists(TRAIN_PATH):
    # Create training generator (with augmentation)
    train_generator = train_datagen.flow_from_directory(
        TRAIN_PATH,
        target_size=(64, 64),
        batch_size=32,
        class_mode='categorical',
        subset='training',
        seed=42,
        shuffle=True
    )
    
    # Create validation generator (no augmentation)
    val_generator = val_datagen.flow_from_directory(
        TRAIN_PATH,
        target_size=(64, 64),
        batch_size=32,
        class_mode='categorical',
        subset='validation',
        seed=42,
        shuffle=False
    )
    
    # Create test generator
    if os.path.exists(TEST_PATH):
        test_generator = test_datagen.flow_from_directory(
            TEST_PATH,
            target_size=(64, 64),
            batch_size=32,
            class_mode='categorical',
            shuffle=False
        )
        print("✓ Test generator created")
    else:
        print("⚠️ Test directory not found, will use validation for evaluation")
        test_generator = None
    
    print(f"✓ Training generator: {train_generator.n} images")
    print(f"✓ Validation generator: {val_generator.n} images")
    print(f"  - Train/Val split: {train_generator.n / (train_generator.n + val_generator.n) * 100:.1f}% / {val_generator.n / (train_generator.n + val_generator.n) * 100:.1f}%")
    print(f"  - Number of classes: {len(train_generator.class_indices)}")
    print(f"  - Batch size: 32")
    print(f"  - Input shape: 64×64×3")

In [ ]:
# 2.2 Setup ImageDataGenerator with Augmentation

# Training data generator WITH augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,                          # Normalize to [0, 1]
    rotation_range=15,                       # ±15° rotation (signs at angles)
    width_shift_range=0.1,                   # ±10% horizontal shift
    height_shift_range=0.1,                  # ±10% vertical shift
    zoom_range=0.1,                          # ±10% zoom (distance variation)
    brightness_range=[0.9, 1.1],             # Brightness variation (lighting)
    horizontal_flip=False,                   # NO horizontal flip! (left ≠ right directional signs)
    vertical_flip=False,                     # NO vertical flip! (unusual in driving)
    fill_mode='nearest'                      # Fill mode for shifted areas
)

# Validation data generator (NO augmentation)
val_datagen = ImageDataGenerator(rescale=1./255)

# Test data generator (NO augmentation)
test_datagen = ImageDataGenerator(rescale=1./255)

print("✓ ImageDataGenerator configurations created")
print("\n📌 AUGMENTATION JUSTIFICATION:")
print("  - Rotation (±15°): Signs viewed at angles from moving vehicle")
print("  - Shift (±10%): Sign position varies within camera frame")
print("  - Zoom (±10%): Distance to sign varies (perspective)")
print("  - Brightness (90-110%): Different lighting (time of day, weather)")
print("  - NO Horizontal Flip: 'Turn Left' ≠ 'Turn Right' (semantic difference!)")
print("  - NO Vertical Flip: Unrealistic in driving scenarios")

## Section 2: Data Preprocessing & Augmentation

### 2.1 Preprocessing Strategy

**Goals:**
1. Standardize input size: 64×64 pixels (balance between computational efficiency and detail capture)
2. Normalize pixel values: [0, 1] (enables faster training and better gradient flow)
3. Split data: 80% training, 20% validation (prevents validation data leakage)

**Key Decisions:**
- Fixed random seed for reproducibility
- Categorical (one-hot) encoding for 43 sign classes
- No test set during preprocessing (kept completely separate for final evaluation)

In [ ]:
# 1.4 Display Sample Images from 9 Different Classes

if os.path.exists(TRAIN_PATH):
    # Select 9 random classes
    selected_classes = np.random.choice(list(class_counts.keys()), size=9, replace=False)
    selected_classes = sorted(selected_classes)
    
    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    axes = axes.flatten()
    
    for idx, class_id in enumerate(selected_classes):
        class_dir = os.path.join(TRAIN_PATH, str(class_id))
        images = os.listdir(class_dir)
        
        # Pick a random image from this class
        random_img_path = os.path.join(class_dir, np.random.choice(images))
        img = cv2.imread(random_img_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Display
        axes[idx].imshow(img_rgb)
        axes[idx].set_title(f'Class {class_id}\\n({class_counts[class_id]} images)', 
                           fontsize=11, fontweight='bold')
        axes[idx].axis('off')
        axes[idx].grid(False)
        
        # Print statistics for this image
        print(f"Class {class_id}: Shape={img_rgb.shape}, Size={class_counts[class_id]}")
    
    plt.suptitle('Sample Traffic Signs from 9 Different Classes\\n(Note: varying sizes, lighting, backgrounds, blur)', 
                fontsize=14, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig('outputs/02_sample_images.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n🔍 OBSERVATIONS:")
    print("  - Images vary in size (not uniform)")
    print("  - Different lighting conditions (shadow, brightness)")
    print("  - Varying backgrounds (road, sky, other signs)")
    print("  - Some images have blur (motion, weather)")
    print("  - Signs at different angles and distances")

In [ ]:
# 1.3 Visualize Class Distribution and Identify Imbalance

if 'df_class_dist' in locals():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    
    # Bar chart of all classes
    ax1.bar(df_class_dist['Class ID'], df_class_dist['Image Count'], color='steelblue', alpha=0.7)
    ax1.set_xlabel('Traffic Sign Class ID', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Number of Images', fontsize=12, fontweight='bold')
    ax1.set_title('Class Distribution Across 43 Sign Categories\n(Reveals Class Imbalance)', 
                  fontsize=13, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3)
    
    # Highlight imbalance
    max_class = df_class_dist.iloc[0]
    min_class = df_class_dist.iloc[-1]
    ax1.bar(max_class['Class ID'], max_class['Image Count'], color='green', alpha=0.7, label='Max')
    ax1.bar(min_class['Class ID'], min_class['Image Count'], color='red', alpha=0.7, label='Min')
    ax1.legend()
    
    # Distribution statistics
    class_counts_list = df_class_dist['Image Count'].values
    ax2.hist(class_counts_list, bins=20, color='coral', edgecolor='black', alpha=0.7)
    ax2.axvline(np.mean(class_counts_list), color='green', linestyle='--', linewidth=2, label=f'Mean: {np.mean(class_counts_list):.0f}')
    ax2.axvline(np.median(class_counts_list), color='orange', linestyle='--', linewidth=2, label=f'Median: {np.median(class_counts_list):.0f}')
    ax2.set_xlabel('Images per Class', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Number of Classes', fontsize=12, fontweight='bold')
    ax2.set_title('Distribution of Class Sizes\n(Shows Severe Imbalance)', fontsize=13, fontweight='bold')
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('outputs/01_class_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # IMPORTANT INSIGHT
    print("\n🔴 CLASS IMBALANCE ANALYSIS:")
    print(f"  - Imbalance ratio (Max/Min): {max_class['Image Count'] / min_class['Image Count']:.1f}x")
    print(f"  - This means class {int(max_class['Class ID'])} has {max_class['Image Count'] / min_class['Image Count']:.1f}x more samples than class {int(min_class['Class ID'])}")
    print(f"  - Risk: Model will be biased towards frequently occurring speed limit signs")
    print(f"  - Solution: Consider class weights, oversampling underrepresented classes, or focal loss")

In [ ]:
# 1.2 Dataset Loading and Structure Inspection

# Define dataset paths (Update these with your GTSRB dataset location)
DATASET_PATH = 'data/'  # Replace with actual GTSRB path
TRAIN_PATH = os.path.join(DATASET_PATH, 'GTSRB_Dataset/Train')
TEST_PATH = os.path.join(DATASET_PATH, 'GTSRB_Dataset/Test')

# Check if dataset exists
if os.path.exists(TRAIN_PATH):
    print(f"✓ Training dataset found at: {TRAIN_PATH}")
    
    # Count images per class
    class_counts = {}
    for class_idx in os.listdir(TRAIN_PATH):
        if os.path.isdir(os.path.join(TRAIN_PATH, class_idx)):
            class_dir = os.path.join(TRAIN_PATH, class_idx)
            class_counts[int(class_idx)] = len(os.listdir(class_dir))
    
    # Display dataset statistics
    total_images = sum(class_counts.values())
    num_classes = len(class_counts)
    
    print(f"\n📊 Dataset Statistics:")
    print(f"  - Total classes: {num_classes}")
    print(f"  - Total training images: {total_images}")
    print(f"  - Max images in a class: {max(class_counts.values())}")
    print(f"  - Min images in a class: {min(class_counts.values())}")
    print(f"  - Average images per class: {total_images / num_classes:.1f}")
    
    # Create DataFrame for analysis
    df_class_dist = pd.DataFrame({
        'Class ID': list(class_counts.keys()),
        'Image Count': list(class_counts.values())
    }).sort_values('Image Count', ascending=False)
    
    print("\n📈 Top 10 Most Represented Classes:")
    print(df_class_dist.head(10).to_string(index=False))
    print("\n📉 Top 10 Least Represented Classes:")
    print(df_class_dist.tail(10).to_string(index=False))
else:
    print(f"⚠️ Dataset not found at {TRAIN_PATH}")
    print("Please download GTSRB from http://benchmark.ini.rub.de/gtsrb_dataset.html")

## Section 1: Dataset Loading & Exploratory Data Analysis (EDA)

### 1.1 Dataset Overview

The **GTSRB (German Traffic Sign Recognition Benchmark)** dataset contains over 50,000 images across 43 traffic sign classes. Traffic signs are categorized as:

- **Red Prohibition Signs**: Speed limits, no entry, stop
- **Blue Information Signs**: Mandatory directions, information
- **Yellow Warning Triangles**: Danger warnings
- **Other**: Speed limit zones, end of restrictions, etc.

We will:
1. Load the dataset and inspect its structure
2. Analyze class distribution and identify imbalance
3. Examine image dimensions and pixel distributions
4. Visualize sample images from each class

In [ ]:
# Import Required Libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications import MobileNetV2, ResNet50

# Evaluation Metrics
from sklearn.metrics import classification_report, confusion_matrix
import time

# Add utils to path
sys.path.append('utils')
from detection import detect_and_crop_sign, draw_bounding_box
from models import build_custom_cnn, build_mobilenetv2, build_resnet50

print("✓ All libraries imported successfully")
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

# ADAS Traffic Sign Detection & Classification Pipeline

## Two-Stage Real-World Computer Vision System

**Project Overview**: This notebook implements a complete two-stage ADAS (Advanced Driver Assistance Systems) pipeline for real-world traffic sign recognition. Unlike academic classification exercises using pre-cropped images, this system processes full road scene images, automatically locates traffic signs, crops them, and classifies them into 43 categories.

### Key Challenges Addressed:
1. **Detection**: Locate traffic signs in full road scenes using colour-based thresholding
2. **Classification**: Classify detected signs into 43 GTSRB categories
3. **Deployment Decision**: Speed vs. Accuracy trade-off analysis for onboard vehicle computers

### Business Relevance:
- **Autonomous Vehicles**: End-to-end perception pipeline identical to Tesla/Waymo/Mobileye systems
- **Smart City**: Automated road sign inventory from raw dashcam footage
- **Navigation/Mapping**: Real-world scene processing for Google Maps/HERE/TomTom
- **Fleet Management**: Dashcam evidence analysis for insurance claims